In [ ]:
# =========================================================
# FINAL T-GCN EVALUATION (CORRELATION KNN + PRICE LAGS 1 & 12)
# =========================================================

import numpy as np
import pandas as pd
import time
import random

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

from statsmodels.stats.diagnostic import acorr_ljungbox

# =========================================================
# CONFIG
# =========================================================
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

# temporal splits
TRAIN_END_DATE = pd.Timestamp("2021-03-31")   # for early-stopping train
VAL_END_DATE   = pd.Timestamp("2022-03-31")   # early-stopping validation end
TEST_START_DATE = pd.Timestamp("2022-04-01")  # final test period start

# best hyperparameters from tuning (update if needed)
WINDOW      = 24
HIDDEN_DIM  = 64
DROPOUT     = 0.0
LR          = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE  = 32
MAX_EPOCHS  = 80
PATIENCE    = 8
MAX_GRAD_NORM = 5.0

# device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# feature lists
continuous_cols = [
    "AverageNeighbourPrice","local_I","area_km2","centroid_x","centroid_y",
    "CoL_distance_km","LA_FE","sdlt_perc_threshold","dwelling_stock",
    "population","ashe_weekly","base_rate","claimant_count_prop",
    "planning_decisions_per_1000","planning_granted_prop",
    "rail_station_entry_exit","GDP","CPIH"
]

categorical_cols = [
    "LMIQuadrant__2","LMIQuadrant__3","LMIQuadrant__4",
    "Region_East of England","Region_London","Region_North East",
    "Region_North West","Region_South East","Region_South West",
    "Region_West Midlands","Region_Yorkshire and The Humber"
]

base_feature_cols = continuous_cols + categorical_cols

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================================
# METRIC FUNCTIONS
# =========================================================
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(100.0 * np.mean(
        2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)
    ))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """
    MASE using seasonal naive of lag m on TRAIN (pre-test) period.
    """
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)

def directional_accuracy(df, entity_col, time_col, y_col, yhat_col):
    """
    Sign accuracy of month-on-month changes, averaged across LAs.
    """
    acc_list = []
    for la, sub in df.groupby(entity_col):
        sub = sub.sort_values(time_col)
        dy_true = sub[y_col].diff()
        dy_pred = sub[yhat_col].diff()
        mask = dy_true.notna() & dy_pred.notna() & (dy_true != 0)
        if mask.sum() == 0:
            continue
        correct = np.sign(dy_true[mask]) == np.sign(dy_pred[mask])
        acc_list.append(correct.mean())
    if not acc_list:
        return np.nan
    return float(np.mean(acc_list))

def morans_i(residuals, xs, ys, k=5):
    """
    Moran's I using k-NN weights on coordinates.
    residuals: [N], xs, ys: [N]
    """
    residuals = np.asarray(residuals)
    xs = np.asarray(xs)
    ys = np.asarray(ys)
    N = len(residuals)
    coords = np.column_stack([xs, ys])

    nbrs = NearestNeighbors(n_neighbors=k+1).fit(coords)
    _, indices = nbrs.kneighbors(coords)

    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        for j in indices[i, 1:]:
            W[i, j] = 1.0
            W[j, i] = 1.0

    S0 = W.sum()
    if S0 == 0:
        return np.nan

    x = residuals
    x_bar = x.mean()
    num = 0.0
    for i in range(N):
        for j in range(N):
            num += W[i, j] * (x[i] - x_bar) * (x[j] - x_bar)
    den = ((x - x_bar) ** 2).sum()
    if den == 0:
        return np.nan

    I = (N / S0) * (num / den)
    return float(I)

# =========================================================
# LOAD DATA & CENTROIDS
# =========================================================
df_full = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df_full = df_full.sort_values([ENTITY_COL, TIME_COL]).reset_index(drop=True)

# robust centroids: ffill/bfill per LA, then take one row per LA
df_full[["centroid_x", "centroid_y"]] = (
    df_full.groupby(ENTITY_COL)[["centroid_x", "centroid_y"]]
           .ffill()
           .bfill()
)

centroid_df = (
    df_full.drop_duplicates(ENTITY_COL)
           .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

# model df: restrict to period from 2007-04 onwards
df = df_full[df_full[TIME_COL] >= pd.Timestamp("2007-04-01")].copy()
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

# LAs present in this period
la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)
print("Number of LAs used:", N)

# align centroids to la_order
centroid_df = centroid_df.loc[la_order]
# if any missing, drop those LAs and filter df accordingly
bad_las = centroid_df[centroid_df.isna().any(axis=1)].index.tolist()
if bad_las:
    print(f"⚠ Dropping {len(bad_las)} LAs with missing centroids:", bad_las)
    centroid_df = centroid_df.dropna()
    la_order = centroid_df.index.tolist()
    df = df[df[ENTITY_COL].isin(la_order)].copy()
    N = len(la_order)
    print("Updated number of LAs:", N)

# =========================================================
# COMPLETE PANEL [T, N], ADD PRICE LAGS
# =========================================================
# ensure one row per (Date, LA)
df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
print("Total time steps:", T_total)

full_index = pd.MultiIndex.from_product(
    [dates, la_order],
    names=[TIME_COL, ENTITY_COL]
)

feature_cols = base_feature_cols.copy()

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# ffill/bfill features + target within each LA over time
df_panel[feature_cols + [TARGET_COL]] = (
    df_panel[feature_cols + [TARGET_COL]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

# add price lags 1 and 12
df_panel["price_lag1"] = (
    df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(1)
)
df_panel["price_lag12"] = (
    df_panel.groupby(level=ENTITY_COL)[TARGET_COL].shift(12)
)

df_panel[["price_lag1", "price_lag12"]] = (
    df_panel[["price_lag1", "price_lag12"]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

lag_price_cols = ["price_lag1", "price_lag12"]
feature_cols = feature_cols + lag_price_cols

# last-resort fill for any remaining NaNs
missing_total = df_panel[feature_cols + [TARGET_COL]].isna().sum().sum()
if missing_total > 0:
    print(f"⚠ {missing_total} NaNs after ffill/bfill. Filling with column means.")
    col_means = df_panel[feature_cols + [TARGET_COL]].mean()
    df_panel[feature_cols + [TARGET_COL]] = df_panel[feature_cols + [TARGET_COL]].fillna(col_means)

print("NaNs after panel completion:",
      df_panel[feature_cols + [TARGET_COL]].isna().sum().sum())

F = len(feature_cols)

# build X_all, y_all in original scale
X_all = (
    df_panel[feature_cols]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N, F)
)
y_all = (
    df_panel[TARGET_COL]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N)
)

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)

# =========================================================
# TIME INDICES: TRAIN / VAL / TEST
# =========================================================
train_end_idx = np.searchsorted(dates, TRAIN_END_DATE, side="right")
val_end_idx   = np.searchsorted(dates, VAL_END_DATE,   side="right")
test_start_idx = np.searchsorted(dates, TEST_START_DATE, side="left")

print(f"Train ends at idx {train_end_idx-1}, date {dates[train_end_idx-1].date()}")
print(f"Val   ends at idx {val_end_idx-1}, date {dates[val_end_idx-1].date()}")
print(f"Test starts at idx {test_start_idx}, date {dates[test_start_idx].date()}")

# =========================================================
# CORRELATION-BASED KNN ADJACENCY (PRE-TEST ONLY)
# =========================================================
# build adjacency using all pre-test prices (up to VAL_END_DATE)
y_corr = y_all[:val_end_idx]   # [T_tv, N]

corr = np.corrcoef(y_corr.T)   # [N, N]
corr = np.nan_to_num(corr, nan=0.0)
np.fill_diagonal(corr, 0.0)

K = 8  # neighbours per node
A = np.zeros((N, N), dtype=np.float32)
for i in range(N):
    nbr_idx = np.argsort(-np.abs(corr[i]))[:K]
    for j in nbr_idx:
        A[i, j] = 1.0
        A[j, i] = 1.0

A = A + np.eye(N, dtype=np.float32)
deg = A.sum(axis=1)
print("Min/Max degree (corr graph):", deg.min(), deg.max())

D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + 1e-8))
A_hat_np = D_inv_sqrt @ A @ D_inv_sqrt
A_hat = torch.tensor(A_hat_np, dtype=torch.float32, device=DEVICE)
print("A_hat shape:", A_hat.shape)

# =========================================================
# SCALING (FIT ON PRE-TEST ONLY) + DATASETS
# =========================================================
# scale on all pre-test data (train + val), no test leakage
X_tv_flat = X_all[:val_end_idx].reshape(-1, F)
y_tv_flat = y_all[:val_end_idx].reshape(-1, 1)

x_scaler = StandardScaler()
X_all_scaled = X_all.copy()
X_all_scaled[:val_end_idx] = x_scaler.fit_transform(X_tv_flat).reshape(-1, N, F)
X_all_scaled[val_end_idx:] = x_scaler.transform(
    X_all[val_end_idx:].reshape(-1, F)
).reshape(-1, N, F)

y_scaler = RobustScaler()
y_all_scaled = y_all.copy()
y_all_scaled[:val_end_idx] = y_scaler.fit_transform(y_tv_flat).reshape(-1, N)
y_all_scaled[val_end_idx:] = y_scaler.transform(
    y_all[val_end_idx:].reshape(-1, 1)
).reshape(-1, N)

y_scale_factor = float(y_scaler.scale_[0])
print("NaNs in X_all_scaled:", np.isnan(X_all_scaled).sum())
print("NaNs in y_all_scaled:", np.isnan(y_all_scaled).sum())

# keep a copy of original y for metrics
y_all_orig = y_all.copy()

# =========================================================
# DATASET CLASS
# =========================================================
class SpatioTemporalDataset(Dataset):
    def __init__(self, X, y, start_t, end_t, window):
        self.X = X
        self.y = y
        self.window = window
        self.indices = [
            t for t in range(start_t, end_t)
            if t - window >= 0
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t = self.indices[idx]
        X_seq = self.X[t - self.window:t]   # [window, N, F]
        y_t   = self.y[t]                  # [N]
        return (
            torch.tensor(X_seq, dtype=torch.float32),
            torch.tensor(y_t,   dtype=torch.float32),
            t
        )

train_ds = SpatioTemporalDataset(X_all_scaled, y_all_scaled,
                                 start_t=WINDOW, end_t=train_end_idx,
                                 window=WINDOW)
val_ds   = SpatioTemporalDataset(X_all_scaled, y_all_scaled,
                                 start_t=train_end_idx, end_t=val_end_idx,
                                 window=WINDOW)
test_ds  = SpatioTemporalDataset(X_all_scaled, y_all_scaled,
                                 start_t=test_start_idx, end_t=T_total,
                                 window=WINDOW)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))
print("Test samples:", len(test_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# =========================================================
# T-GCN MODEL
# =========================================================
class GraphConv(nn.Module):
    def __init__(self, in_feats, out_feats):
        super().__init__()
        self.linear = nn.Linear(in_feats, out_feats)

    def forward(self, X, A_hat):
        return self.linear(torch.einsum("ij,bjf->bif", A_hat, X))

class TGCNCell(nn.Module):
    def __init__(self, in_feats, hidden_dim, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gc_zr = GraphConv(in_feats + hidden_dim, 2 * hidden_dim)
        self.gc_h  = GraphConv(in_feats + hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X_t, H_prev, A_hat):
        if H_prev is None:
            H_prev = torch.zeros(
                X_t.size(0), X_t.size(1), self.hidden_dim,
                device=X_t.device
            )
        XH = torch.cat([X_t, H_prev], dim=-1)

        ZR = torch.sigmoid(self.gc_zr(XH, A_hat))
        Z, R = torch.chunk(ZR, 2, dim=-1)

        XH_candidate = torch.cat([X_t, R * H_prev], dim=-1)
        H_tilde = torch.tanh(self.gc_h(XH_candidate, A_hat))

        H_new = (1 - Z) * H_prev + Z * H_tilde
        H_new = self.dropout(H_new)
        return H_new

class TGCN(nn.Module):
    def __init__(self, num_nodes, in_feats, hidden_dim, dropout=0.0):
        super().__init__()
        self.cell = TGCNCell(in_feats, hidden_dim, dropout=dropout)
        self.out  = nn.Linear(hidden_dim, 1)

    def forward(self, X_seq, A_hat):
        B, T_seq, N_nodes, F_in = X_seq.shape
        H = None
        for t in range(T_seq):
            X_t = X_seq[:, t]
            H   = self.cell(X_t, H, A_hat)
        y_hat = self.out(H).squeeze(-1)
        return y_hat

# =========================================================
# TRAIN WITH EARLY STOPPING
# =========================================================
model = TGCN(num_nodes=N, in_feats=F, hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
mse_loss = nn.MSELoss()

best_val_mse = np.inf
best_epoch   = -1
epochs_no_improve = 0
best_state = None

print("\n=== Training final T-GCN with early stopping ===")
start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    batch_losses = []

    for X_seq, y_t, t_idx in train_loader:
        X_seq = X_seq.to(DEVICE)
        y_t   = y_t.to(DEVICE)

        optimizer.zero_grad()
        y_hat = model(X_seq, A_hat)
        loss  = mse_loss(y_hat, y_t)

        if not torch.isfinite(loss):
            print(f"  ⚠ Non-finite training loss at epoch {epoch}. Aborting training.")
            break

        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        batch_losses.append(loss.item())

    if not batch_losses:
        break

    # validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for X_seq, y_t, t_idx in val_loader:
            X_seq = X_seq.to(DEVICE)
            y_t   = y_t.to(DEVICE)
            y_hat = model(X_seq, A_hat)
            vloss = mse_loss(y_hat, y_t)
            if torch.isfinite(vloss):
                val_losses.append(vloss.item())

    if not val_losses:
        print("  ⚠ All val losses non-finite; stopping.")
        break

    val_mse = float(np.mean(val_losses))
    val_rmse_orig = np.sqrt(val_mse) * y_scale_factor

    print(f"Epoch {epoch:03d} | "
          f"train MSE={np.mean(batch_losses):.4f} | "
          f"val MSE={val_mse:.4f} | "
          f"val RMSE(£)={val_rmse_orig:,.1f}")

    if val_mse + 1e-6 < best_val_mse:
        best_val_mse = val_mse
        best_epoch   = epoch
        epochs_no_improve = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

train_time = time.time() - start_time
print(f"Training finished in {train_time:.1f} seconds. Best epoch: {best_epoch}")

# load best state
if best_state is not None:
    model.load_state_dict(best_state)
else:
    print("⚠ No best_state captured; using last epoch weights.")

# =========================================================
# TEST EVALUATION
# =========================================================
model.eval()
y_true_batches = []
y_pred_batches = []
t_index_batches = []

with torch.no_grad():
    for X_seq, y_t, t_idx in test_loader:
        X_seq = X_seq.to(DEVICE)
        y_t   = y_t.to(DEVICE)
        y_hat = model(X_seq, A_hat)

        y_true_batches.append(y_t.cpu().numpy())   # scaled
        y_pred_batches.append(y_hat.cpu().numpy())
        t_index_batches.append(np.array(t_idx))

y_true_scaled = np.concatenate(y_true_batches, axis=0)  # [S, N]
y_pred_scaled = np.concatenate(y_pred_batches, axis=0)  # [S, N]
t_indices     = np.concatenate(t_index_batches, axis=0) # [S]

# sort by time index just in case
order = np.argsort(t_indices)
t_sorted = t_indices[order]
y_true_scaled_sorted = y_true_scaled[order]
y_pred_scaled_sorted = y_pred_scaled[order]

# inverse-transform to original £
y_true_flat_scaled = y_true_scaled_sorted.reshape(-1, 1)
y_pred_flat_scaled = y_pred_scaled_sorted.reshape(-1, 1)

y_true_flat_orig = y_scaler.inverse_transform(y_true_flat_scaled).reshape(-1, N)
y_pred_flat_orig = y_scaler.inverse_transform(y_pred_flat_scaled).reshape(-1, N)

# build DataFrame with one row per (time, LA)
S = len(t_sorted)
dates_rep = np.repeat(dates[t_sorted], N)
la_rep    = np.tile(la_order, S)

df_test = pd.DataFrame({
    TIME_COL: dates_rep,
    ENTITY_COL: la_rep,
    "y_true": y_true_flat_orig.reshape(-1),
    "y_pred": y_pred_flat_orig.reshape(-1),
})
df_test["resid"] = df_test["y_true"] - df_test["y_pred"]

# For MASE scaling: use all pre-test y (train+val) flattened
y_train_for_mase = y_all_orig[:val_end_idx].reshape(-1)

# =========================================================
# GLOBAL METRICS
# =========================================================
global_mae  = mae(df_test["y_true"], df_test["y_pred"])
global_rmse = rmse(df_test["y_true"], df_test["y_pred"])
global_smape = smape(df_test["y_true"], df_test["y_pred"])
global_mase  = mase(df_test["y_true"], df_test["y_pred"], y_train_for_mase, m=12)

print("\n=== Global accuracy ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:.3f}%")
print(f"MASE  : {global_mase:.3f}")

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae = (
    df_test.groupby(ENTITY_COL)
           .apply(lambda g: mae(g["y_true"], g["y_pred"]))
)

median_mae = float(np.median(la_mae.values))
p75_mae    = float(np.percentile(la_mae.values, 75))

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
# Moran's I: average residual per LA
la_resid_mean = df_test.groupby(ENTITY_COL)["resid"].mean()
centroids = centroid_df.loc[la_resid_mean.index][["centroid_x", "centroid_y"]]

I_moran = morans_i(
    residuals=la_resid_mean.values,
    xs=centroids["centroid_x"].values,
    ys=centroids["centroid_y"].values,
    k=5
)

# Ljung–Box on mean residual over time
monthly_resid = (
    df_test.groupby(TIME_COL)["resid"]
           .mean()
           .sort_index()
)

lb = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
lb_stat = float(lb["lb_stat"].iloc[0])
lb_p    = float(lb["lb_pvalue"].iloc[0])

print("\n=== Spatio-temporal diagnostics ===")
print(f"Moran's I (mean residuals across LAs): {I_moran:.4f}")
print(f"Ljung–Box Q(12): stat={lb_stat:.3f}, p={lb_p:.4f}")

# =========================================================
# DIRECTIONAL ACCURACY & GROWTH-RATE ERROR
# =========================================================
dir_acc = directional_accuracy(df_test, ENTITY_COL, TIME_COL, "y_true", "y_pred")

# approximate 12-month growth-rate error
# build full predicted panel aligned in time for test months
y_pred_full = np.full_like(y_all_orig, np.nan, dtype=np.float32)
for j, t in enumerate(t_sorted):
    y_pred_full[t] = y_pred_flat_orig[j]

errs = []
for t in range(test_start_idx, T_total):
    t_prev = t - 12
    if t_prev < 0:
        continue
    true_t   = y_all_orig[t]      # [N]
    true_prev= y_all_orig[t_prev] # [N]
    pred_t   = y_pred_full[t]     # [N]
    mask = (~np.isnan(pred_t)) & (true_t > 0) & (true_prev > 0)
    if not mask.any():
        continue
    true_growth = np.log(true_t[mask]) - np.log(true_prev[mask])
    pred_growth = np.log(pred_t[mask]) - np.log(true_prev[mask])
    errs.append(np.abs(true_growth - pred_growth))

if errs:
    gre_mae = float(np.mean(np.concatenate(errs)))
else:
    gre_mae = np.nan

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")
